In [1]:
import torch
import torch.nn.functional as F

# Import your project modules
import config as cfg
import cnn_utils.data as data_utils
from cnn_utils.model import load_model
from cnn_utils.evaluate import EvalClassification
from cnn_utils.patching import get_dataloader as get_patch_dataloader

# 1. Load the Phase 6 Linear Baseline Model
# (Change to a Phase 7 checkpoint if your Gated Attention numbers were from Phase 7)
MODEL_PATH = cfg.ph6_linear_best 
print(f"Loading model from: {MODEL_PATH}")
model_linear = load_model(MODEL_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_linear.to(device)
model_linear.eval()

# 2. Load the Dataloaders
print("Loading Lab Dataloaders...")
_, _, test_dl_lab = data_utils.get_3_dataloaders(cfg)

print("Loading OOD Dataloaders...")
_, test_dl_ood = data_utils.get_ood_dataloaders(cfg)

# 3. Initialize Evaluators
eval_lab = EvalClassification(cfg, model_linear, test_dl_lab)
eval_ood = EvalClassification(cfg, model_linear, test_dl_ood)

/home/takayuki/.local/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(
/home/takayuki/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/home/takayuki/.local/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Outp

Loading model from: /home/takayuki/Desktop/summer2025/plants/training/phase6/models/best_model_07-14_14-56--10_p6_linear_convnextv2_tiny.fcmae_ft_in22k_in1k.pth
Model loaded successfully from /home/takayuki/Desktop/summer2025/plants/training/phase6/models/best_model_07-14_14-56--10_p6_linear_convnextv2_tiny.fcmae_ft_in22k_in1k.pth
Loading Lab Dataloaders...
 Total dataset size 	: 230
 Train dataset size 	: 138
 Val dataset size 	: 46
 Test dataset size 	: 46

Loading OOD Dataloaders...
Found 10 classes locally: ['Hydrocharis morsus-ranae', 'Myriophyllum spicatum', 'Nitellopsis obtusa', 'Nuphar variegata', 'Potamogeton crispus', 'Potamogeton gramineus', 'Potamogeton illinoensis', 'Potamogeton richardsonii', 'Potamogeton robbinsii', 'Ranunculus aquatilis']

OOD split complete:
  OOD Validation set size: 16
  OOD Test set size    : 17



In [2]:
# Cell 2
# 1. Load the Phase 7 Gated Attention Model
GATED_MODEL_PATH = cfg.ph7_gated_attn_best_ood_acc # Or ph7_gated_attn_best_ood_loss based on your final champion
print(f"Loading Gated Attention model from: {GATED_MODEL_PATH}")

model_gated = load_model(GATED_MODEL_PATH)
model_gated.to(device)
model_gated.eval()

# 2. Initialize Evaluators for Gated Attention
eval_lab_gated = EvalClassification(cfg, model_gated, test_dl_lab)
eval_ood_gated = EvalClassification(cfg, model_gated, test_dl_ood)

Loading Gated Attention model from: /home/takayuki/Desktop/summer2025/plants/training/phase7/models/best_ood_acc_model_07-31_04-19--55_ph7_gated_attn_actual_convnextv2_tiny.fcmae_ft_in22k_in1k.pth
Model loaded successfully from /home/takayuki/Desktop/summer2025/plants/training/phase7/models/best_ood_acc_model_07-31_04-19--55_ph7_gated_attn_actual_convnextv2_tiny.fcmae_ft_in22k_in1k.pth


In [3]:
# Cell 3
# Dynamically override the configuration variables for patching and YNLT

# 1. Patching Parameters
cfg.patching_method = 'multi-scale'
cfg.padding_logic = 'reflect'

# Define the scales and overlaps you want to experiment with
cfg.ms_scale = [0.2, 0.3, 0.4, 0.5] 
cfg.ms_overlap = [0.0, 0.1, 0.2, 0.3] 

# 2. Confusion Subsystem Parameters
cfg.do_critical_review = True

cfg.critical_confusion_level = 1       # Threshold to trigger YNLT

# cfg.margin_threshold = 0.2
# cfg.entropy_threshold = 2.0
# cfg.confidence_threshold = 0.5

cfg.margin_threshold = 0.5      # Trigger if top 2 predictions are within 50% of each other (was 0.2)
cfg.confidence_threshold = 0.8  # Trigger if top prediction is under 80% confident (was 0.5)
cfg.entropy_threshold = 1.0     # Trigger if probability spread is wider (was 2.0)

cfg.ignore_invasive_predictions = True # As per your config, avoid double-checking if already flagged invasive

print("YNLT and Patching Hyperparameters configured:")
print(f"Method: {cfg.patching_method}")
print(f"Scales: {cfg.ms_scale}")
print(f"Overlaps: {cfg.ms_overlap}")

YNLT and Patching Hyperparameters configured:
Method: multi-scale
Scales: [0.2, 0.3, 0.4, 0.5]
Overlaps: [0.0, 0.1, 0.2, 0.3]


In [4]:
# Cell 4
from cnn_utils.predict import PredictPlants # Adjust import path if needed based on your notebook location

# Initialize the YNLT Predictors
predictor_linear = PredictPlants(model_linear, "Linear Baseline")
predictor_gated = PredictPlants(model_gated, "Gated Attention")

# Create wrapper functions to match the signature expected by evaluator.evaluate()
# EvalClassification passes (images, label_item) to the predict_function
def predict_fn_linear_ynlt(images, label):
    return predictor_linear.predict(input=images, true_label=label, return_fig=False, display=False)

def predict_fn_gated_ynlt(images, label):
    return predictor_gated.predict(input=images, true_label=label, return_fig=False, display=False)

In [5]:
# Cell 5
import pandas as pd

print("Running Ablation on LAB Test Set...\n")

# 1. Linear Classifier (Base)
print("Evaluating: Linear Classifier")
eval_lab.evaluate()
lin_lab_metrics = eval_lab.get_binary_metrics(display=False)

# 2. Linear + YNLT
print("Evaluating: Linear Classifier + YNLT")
eval_lab.evaluate(predict_function=predict_fn_linear_ynlt)
lin_ynlt_lab_metrics = eval_lab.get_binary_metrics(display=False)

# 3. Gated Attention (Base)
print("Evaluating: Gated Attention")
eval_lab_gated.evaluate()
gat_lab_metrics = eval_lab_gated.get_binary_metrics(display=False)

# 4. Gated Attention + YNLT
print("Evaluating: Gated Attention + YNLT")
eval_lab_gated.evaluate(predict_function=predict_fn_gated_ynlt)
gat_ynlt_lab_metrics = eval_lab_gated.get_binary_metrics(display=False)

# Compile Lab Results
lab_results = pd.DataFrame([
    {"Model": "Linear Classifier", "Lab Acc (%)": lin_lab_metrics['Accuracy']*100, "Lab FNR (%)": lin_lab_metrics['FNR']*100},
    {"Model": "Linear + YNLT", "Lab Acc (%)": lin_ynlt_lab_metrics['Accuracy']*100, "Lab FNR (%)": lin_ynlt_lab_metrics['FNR']*100},
    {"Model": "Gated Attention", "Lab Acc (%)": gat_lab_metrics['Accuracy']*100, "Lab FNR (%)": gat_lab_metrics['FNR']*100},
    {"Model": "Gated Attention + YNLT", "Lab Acc (%)": gat_ynlt_lab_metrics['Accuracy']*100, "Lab FNR (%)": gat_ynlt_lab_metrics['FNR']*100}
])

print("\n--- Lab Data Results ---")
print(lab_results.round(2).to_string(index=False))

Running Ablation on LAB Test Set...

Evaluating: Linear Classifier


  -- Evaluating: 100%|██████████| 46/46 [00:03<00:00, 12.06it/s]


Evaluating: Linear Classifier + YNLT


  -- Evaluating: 100%|██████████| 46/46 [00:20<00:00,  2.28it/s]


Evaluating: Gated Attention


  -- Evaluating: 100%|██████████| 46/46 [00:05<00:00,  8.95it/s]


Evaluating: Gated Attention + YNLT


  -- Evaluating: 100%|██████████| 46/46 [00:15<00:00,  2.92it/s]


--- Lab Data Results ---
                 Model  Lab Acc (%)  Lab FNR (%)
     Linear Classifier        93.48         30.0
         Linear + YNLT        93.48         30.0
       Gated Attention        97.83         10.0
Gated Attention + YNLT        97.83         10.0


In [6]:
# Cell 6
print("Running Ablation on OOD Test Set...\n")

# 1. Linear Classifier (Base)
print("Evaluating: Linear Classifier")
eval_ood.evaluate()
lin_ood_metrics = eval_ood.get_binary_metrics(display=False)

# 2. Linear + YNLT
print("Evaluating: Linear Classifier + YNLT")
eval_ood.evaluate(predict_function=predict_fn_linear_ynlt)
lin_ynlt_ood_metrics = eval_ood.get_binary_metrics(display=False)

# 3. Gated Attention (Base)
print("Evaluating: Gated Attention")
eval_ood_gated.evaluate()
gat_ood_metrics = eval_ood_gated.get_binary_metrics(display=False)

# 4. Gated Attention + YNLT
print("Evaluating: Gated Attention + YNLT")
eval_ood_gated.evaluate(predict_function=predict_fn_gated_ynlt)
gat_ynlt_ood_metrics = eval_ood_gated.get_binary_metrics(display=False)

# Compile OOD Results
ood_results = pd.DataFrame([
    {"Model": "Linear Classifier", "OOD Acc (%)": lin_ood_metrics['Accuracy']*100, "OOD FNR (%)": lin_ood_metrics['FNR']*100},
    {"Model": "Linear + YNLT", "OOD Acc (%)": lin_ynlt_ood_metrics['Accuracy']*100, "OOD FNR (%)": lin_ynlt_ood_metrics['FNR']*100},
    {"Model": "Gated Attention", "OOD Acc (%)": gat_ood_metrics['Accuracy']*100, "OOD FNR (%)": gat_ood_metrics['FNR']*100},
    {"Model": "Gated Attention + YNLT", "OOD Acc (%)": gat_ynlt_ood_metrics['Accuracy']*100, "OOD FNR (%)": gat_ynlt_ood_metrics['FNR']*100}
])

print("\n--- Out-Of-Distribution (OOD) Data Results ---")
print(ood_results.round(2).to_string(index=False))

Running Ablation on OOD Test Set...

Evaluating: Linear Classifier


  -- Evaluating: 100%|██████████| 17/17 [00:01<00:00, 13.44it/s]


Evaluating: Linear Classifier + YNLT


  -- Evaluating: 100%|██████████| 17/17 [00:20<00:00,  1.22s/it]


Evaluating: Gated Attention


  -- Evaluating: 100%|██████████| 17/17 [00:01<00:00,  9.87it/s]


Evaluating: Gated Attention + YNLT


  -- Evaluating: 100%|██████████| 17/17 [00:10<00:00,  1.65it/s]


--- Out-Of-Distribution (OOD) Data Results ---
                 Model  OOD Acc (%)  OOD FNR (%)
     Linear Classifier        58.82         62.5
         Linear + YNLT        58.82         62.5
       Gated Attention        76.47         12.5
Gated Attention + YNLT        76.47         12.5


In [7]:
# Quick diagnostic to see if YNLT is triggering
trigger_count = 0
total_images = len(test_dl_ood.dataset)

for images, labels in test_dl_ood:
    images = images.to(device)
    # Get the raw prediction dictionary (with display=False to avoid spamming the output)
    pred_dict = predictor_gated.predict(input=images, true_label=labels.item(), return_fig=False, display=False)
    
    if pred_dict.get("is_critical", False):
        trigger_count += 1

print(f"YNLT triggered on {trigger_count} out of {total_images} OOD images.")

YNLT triggered on 4 out of 17 OOD images.


In [8]:
# Cell 7: Inspect YNLT Decisions
print("Inspecting YNLT overrides on the OOD Test Set...\n")

for i, (images, labels) in enumerate(test_dl_ood):
    images = images.to(device)
    true_label_idx = labels.item()
    true_name = cfg.CANONICAL_INDEX_TO_CLASS[true_label_idx]
    
    # Run prediction
    pred_dict = predictor_gated.predict(input=images, true_label=true_label_idx, return_fig=False, display=False)
    
    # Check if YNLT triggered
    if pred_dict.get("is_critical", False):
        info = pred_dict.get("critical_review_info", {})
        init_pred = info.get("initial_prediction_name")
        final_pred = info.get("final_prediction_name")
        
        print(f"Image {i} | True Label: {true_name}")
        print(f"  -> Initial Pred : {init_pred} {'(Correct)' if init_pred == true_name else '(Wrong)'}")
        print(f"  -> YNLT Pred    : {final_pred} {'(Correct)' if final_pred == true_name else '(Wrong)'}")
        print("-" * 50)

Inspecting YNLT overrides on the OOD Test Set...

Image 5 | True Label: Potamogeton richardsonii
  -> Initial Pred : Potamogeton natans (Wrong)
  -> YNLT Pred    : Potamogeton illinoensis (Wrong)
--------------------------------------------------
Image 7 | True Label: Potamogeton illinoensis
  -> Initial Pred : Vallisneria americana (Wrong)
  -> YNLT Pred    : Potamogeton illinoensis (Correct)
--------------------------------------------------
Image 8 | True Label: Nuphar variegata
  -> Initial Pred : Nuphar variegata (Correct)
  -> YNLT Pred    : Nuphar variegata (Correct)
--------------------------------------------------
Image 16 | True Label: Potamogeton gramineus
  -> Initial Pred : Heteranthera dubia (Wrong)
  -> YNLT Pred    : Potamogeton gramineus (Correct)
--------------------------------------------------


In [9]:
# Cell 6 (Updated): Validate OOD Test Set Results with Class-wise Accuracy
print("Running Ablation on OOD Test Set (Class-wise)...\n")

# 1. Linear Classifier (Base)
eval_ood.evaluate()
lin_ood_acc = eval_ood.get_accuracy(verbose=False)
lin_ood_fnr = eval_ood.get_binary_metrics(display=False)['FNR']

# 2. Linear + YNLT
eval_ood.evaluate(predict_function=predict_fn_linear_ynlt)
lin_ynlt_ood_acc = eval_ood.get_accuracy(verbose=False)
lin_ynlt_ood_fnr = eval_ood.get_binary_metrics(display=False)['FNR']

# 3. Gated Attention (Base)
eval_ood_gated.evaluate()
gat_ood_acc = eval_ood_gated.get_accuracy(verbose=False)
gat_ood_fnr = eval_ood_gated.get_binary_metrics(display=False)['FNR']

# 4. Gated Attention + YNLT
eval_ood_gated.evaluate(predict_function=predict_fn_gated_ynlt)
gat_ynlt_ood_acc = eval_ood_gated.get_accuracy(verbose=False)
gat_ynlt_ood_fnr = eval_ood_gated.get_binary_metrics(display=False)['FNR']

# Compile Results
ood_results = pd.DataFrame([
    {"Model": "Linear Classifier", "OOD Class Acc (%)": lin_ood_acc*100, "OOD Binary FNR (%)": lin_ood_fnr*100},
    {"Model": "Linear + YNLT", "OOD Class Acc (%)": lin_ynlt_ood_acc*100, "OOD Binary FNR (%)": lin_ynlt_ood_fnr*100},
    {"Model": "Gated Attention", "OOD Class Acc (%)": gat_ood_acc*100, "OOD Binary FNR (%)": gat_ood_fnr*100},
    {"Model": "Gated Attention + YNLT", "OOD Class Acc (%)": gat_ynlt_ood_acc*100, "OOD Binary FNR (%)": gat_ynlt_ood_fnr*100}
])

print("\n--- Out-Of-Distribution (OOD) Data Results ---")
print(ood_results.round(2).to_string(index=False))

Running Ablation on OOD Test Set (Class-wise)...



  -- Evaluating: 100%|██████████| 17/17 [00:09<00:00,  1.82it/s]


--- Out-Of-Distribution (OOD) Data Results ---
                 Model  OOD Class Acc (%)  OOD Binary FNR (%)
     Linear Classifier              47.06                62.5
         Linear + YNLT              52.94                62.5
       Gated Attention              52.94                12.5
Gated Attention + YNLT              64.71                12.5
